# LeetCode 2119 — A Number After a Double Reversal bằng Python

**Link đề:** https://leetcode.com/problems/a-number-after-a-double-reversal/
**Độ khó:** Easy · **Chủ đề:** Math

```python
class Solution:
    def isSameAfterReversals(self, num: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu phép **đảo chữ số** của một số nguyên và vì sao **số 0 ở đầu bị mất** (`12300 → 321`).
2. Đi từ **mô phỏng đúng như đề** (đảo hai lần bằng chuỗi hoặc bằng toán `% 10`, `// 10`) → **quan sát toán học**: chỉ cần kiểm tra chữ số cuối, `O(1)`.
3. Nắm cú pháp: `str()`, `int()`, slicing `[::-1]`, `%`, `//`, `while`, `or`, `str.endswith`.
4. Chứng minh ngắn, Time/Space, kiểm thử vét cạn toàn miền, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và cách hiểu đúng

**Đảo** một số nguyên = đảo thứ tự các chữ số, **bỏ các số 0 ở đầu** kết quả.

- Đảo `2021` → `1202`.
- Đảo `12300` → `00321` → `321`.

Cho `num`. Đảo `num` được `reversed1`; đảo `reversed1` được `reversed2`. Trả `True` nếu `reversed2 == num`.

| `num` | reversed1 | reversed2 | Kết quả |
|:---:|:---:|:---:|:---:|
| 526 | 625 | 526 | `True` |
| 1800 | 81 | 18 | `False` |
| 0 | 0 | 0 | `True` |

**Constraints:** `0 <= num <= 10^6`.

In [1]:
examples = [(526, True), (1800, False), (0, True)]
for num, exp in examples:
    print(num, "->", exp)

526 -> True
1800 -> False
0 -> True


## 2. Nền tảng Python cần dùng

### 2.1. `str(num)` và `int(s)` — đổi qua lại giữa số và chuỗi

`int("00321")` tự **bỏ số 0 ở đầu** → `321`. Đây chính là quy tắc "không giữ số 0 đầu" của đề!

In [2]:
print(str(12300), type(str(12300)))
print(int("00321"), int("0"), int("000"))

12300 <class 'str'>
321 0 0


### 2.2. Đảo chuỗi `s[::-1]`

`s[start:end:step]`, bước `-1` đi ngược từ cuối về đầu.

In [3]:
s = "12300"
print(s[::-1])          # '00321'
print(int(s[::-1]))     # 321

00321
321


### 2.3. `% 10` và `// 10` — lấy và bỏ chữ số cuối

- `x % 10`: chữ số hàng đơn vị.
- `x // 10`: bỏ chữ số hàng đơn vị (chia nguyên).
- `x / 10`: chia **thực**, ra số thập phân — **không** dùng khi xử lý chữ số.

In [4]:
x = 1800
print(x % 10, x // 10, x / 10)

0 180 180.0


### 2.4. Đảo số bằng toán: `rev = rev * 10 + x % 10`

Mỗi vòng: lấy chữ số cuối của `x` gắn vào **cuối** `rev` (nhân 10 để "dịch trái" rồi cộng). Ví dụ `526`: `0 → 6 → 62 → 625`.

In [5]:
x, rev = 526, 0
while x > 0:
    d = x % 10
    rev = rev * 10 + d
    x //= 10
    print(f"lấy {d}: rev = {rev}, x còn {x}")

lấy 6: rev = 6, x còn 52
lấy 2: rev = 62, x còn 5
lấy 5: rev = 625, x còn 0


### 2.5. Toán tử logic `or` và `and`

`A or B` đúng khi ít nhất một vế đúng. Python **ngắn mạch**: nếu `A` đúng thì không tính `B`.

In [6]:
for num in [0, 5, 10, 120, 121]:
    print(num, "->", num == 0 or num % 10 != 0)

0 -> True
5 -> True
10 -> False
120 -> False
121 -> True


### 2.6. "Truthiness" — giá trị đúng/sai của số và chuỗi

Trong `if` / `while`, Python tự đổi giá trị sang `bool`:

| Giá trị | `bool(...)` |
|---|---|
| `0`, `0.0` | `False` |
| mọi số khác 0 | `True` |
| `""` (chuỗi rỗng) | `False` |
| `"0"` (chuỗi có một ký tự) | `True` ← dễ nhầm! |

Vì vậy `while x:` dừng khi `x` về 0, nhưng `if "0":` vẫn chạy.

In [7]:
for v in [0, 7, -3, "", "0", "abc"]:
    print(repr(v), "->", bool(v))

0 -> False
7 -> True
-3 -> True
'' -> False
'0' -> True
'abc' -> True


## 3. Cách 1 — Trực giác: làm đúng như đề (đảo hai lần bằng chuỗi)

In [8]:
def reverse_str(x: int) -> int:
    return int(str(x)[::-1])

def is_same_simulate_str(num: int) -> bool:
    r1 = reverse_str(num)
    r2 = reverse_str(r1)
    return r2 == num

for num, exp in examples:
    print(num, "->", reverse_str(num), "->", reverse_str(reverse_str(num)), "|", is_same_simulate_str(num), "kỳ vọng", exp)

526 -> 625 -> 526 | True kỳ vọng True
1800 -> 81 -> 18 | False kỳ vọng False
0 -> 0 -> 0 | True kỳ vọng True


### Cách 1b — Mô phỏng bằng toán (không dùng chuỗi)

Chú ý: với `x = 0` vòng `while x > 0` không chạy, `rev = 0` — vẫn đúng vì đảo 0 là 0.

In [9]:
def reverse_math(x: int) -> int:
    rev = 0
    while x > 0:
        rev = rev * 10 + x % 10
        x //= 10
    return rev

def is_same_simulate_math(num: int) -> bool:
    return reverse_math(reverse_math(num)) == num

for num, exp in examples:
    print(num, is_same_simulate_math(num), "kỳ vọng", exp)

526 True kỳ vọng True
1800 False kỳ vọng False
0 True kỳ vọng True


## 4. Nút thắt — có cần đảo thật không?

Mô phỏng tốn `O(d)` với `d` là số chữ số (≤ 7). Đã đủ nhanh! Nhưng câu hỏi hay hơn: **khi nào** đảo hai lần làm **mất thông tin**?

Thử nhiều số và quan sát những số **thất bại**:

In [10]:
fails = [n for n in range(0, 200) if not is_same_simulate_str(n)]
print("Các số thất bại trong [0, 200):", fails)
print("Mọi số thất bại đều tận cùng bằng 0?", all(n % 10 == 0 for n in fails))
print("Số 0 có thất bại không?", 0 in fails)

Các số thất bại trong [0, 200): [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190]
Mọi số thất bại đều tận cùng bằng 0? True
Số 0 có thất bại không? False


## 5. Bước đột phá — chỉ số 0 ở **cuối** bị mất

Gợi ý của đề: *"Other than the number 0 itself, any number that ends with 0 would lose some digits permanently when reversed."*

Lý luận:
- Đảo `num` đưa các chữ số **cuối** lên **đầu**. Nếu `num` tận cùng bằng 0 (và `num ≠ 0`), các số 0 đó trở thành **số 0 đầu** của `reversed1` → **bị xóa vĩnh viễn**. Đảo lại không thể sinh chúng ra nữa ⇒ `reversed2 < num`.
- Nếu `num` **không** tận cùng bằng 0: chữ số đầu của `reversed1` khác 0 ⇒ không mất chữ số nào; `reversed1` có đủ `d` chữ số. Đảo lại chuỗi đó được đúng chuỗi của `num` (đảo một chuỗi hai lần trả về chính nó), mà chuỗi này bắt đầu khác 0 nên `int` giữ nguyên ⇒ `reversed2 = num`.
- `num = 0`: đảo vẫn là 0 ⇒ `True`.

Kết luận:

```python
return num == 0 or num % 10 != 0
```

In [11]:
for num in [526, 1800, 0, 10, 7, 1000000, 999999]:
    s = str(num)
    print(f"{num:>8}: chuỗi đảo = {s[::-1]!r:10} -> int = {int(s[::-1]):>7} | số 0 cuối bị mất? {num != 0 and num % 10 == 0}")

     526: chuỗi đảo = '625'      -> int =     625 | số 0 cuối bị mất? False
    1800: chuỗi đảo = '0081'     -> int =      81 | số 0 cuối bị mất? True
       0: chuỗi đảo = '0'        -> int =       0 | số 0 cuối bị mất? False
      10: chuỗi đảo = '01'       -> int =       1 | số 0 cuối bị mất? True
       7: chuỗi đảo = '7'        -> int =       7 | số 0 cuối bị mất? False
 1000000: chuỗi đảo = '0000001'  -> int =       1 | số 0 cuối bị mất? True
  999999: chuỗi đảo = '999999'   -> int =  999999 | số 0 cuối bị mất? False


## 6. Lời giải chính — mổ xẻ rồi ghép

### 6.1. Đoạn 1: trường hợp đặc biệt `num == 0`

`0 % 10 == 0` nên nếu chỉ kiểm tra "tận cùng bằng 0" ta sẽ trả sai `False` cho `0`. Phải tách riêng.

In [12]:
num = 0
print("0 % 10 =", num % 10, "-> nếu chỉ kiểm tra num % 10 != 0 sẽ ra", num % 10 != 0, "(sai)")

0 % 10 = 0 -> nếu chỉ kiểm tra num % 10 != 0 sẽ ra False (sai)


### 6.2. Đoạn 2: kiểm tra chữ số cuối

In [13]:
for num in [526, 1800, 120, 121]:
    print(num, "chữ số cuối:", num % 10, "-> giữ được sau 2 lần đảo?", num % 10 != 0)

526 chữ số cuối: 6 -> giữ được sau 2 lần đảo? True
1800 chữ số cuối: 0 -> giữ được sau 2 lần đảo? False
120 chữ số cuối: 0 -> giữ được sau 2 lần đảo? False
121 chữ số cuối: 1 -> giữ được sau 2 lần đảo? True


### 6.3. Ghép lại

In [14]:
def is_same_after_reversals(num: int) -> bool:
    return num == 0 or num % 10 != 0

for num, exp in examples:
    print(num, is_same_after_reversals(num), "kỳ vọng", exp)

526 True kỳ vọng True
1800 False kỳ vọng False
0 True kỳ vọng True


## 7. TRACE từng bước

### 7.1. Bảng trace mô phỏng bằng toán — `num = 1800`

**Lần đảo 1** (`x = 1800`):

| Vòng | x % 10 | rev sau | x sau |
|:---:|:---:|:---:|:---:|
| 1 | 0 | 0 | 180 |
| 2 | 0 | 0 | 18 |
| 3 | 8 | 8 | 1 |
| 4 | 1 | 81 | 0 |

Hai số 0 đầu tiên làm `rev` vẫn bằng 0 — chúng "biến mất". **Lần đảo 2** (`x = 81`) cho `18 ≠ 1800`.

In [15]:
def trace_reverse(x):
    rev, step = 0, 1
    print(f"  đảo {x}:")
    while x > 0:
        d = x % 10
        rev = rev * 10 + d
        x //= 10
        print(f"    vòng {step}: chữ số {d} -> rev={rev}, x={x}")
        step += 1
    return rev

def trace_double(num):
    r1 = trace_reverse(num)
    r2 = trace_reverse(r1)
    print(f"  => reversed1={r1}, reversed2={r2}, so với num={num}: {r2 == num}")
    return r2 == num

trace_double(1800)

  đảo 1800:
    vòng 1: chữ số 0 -> rev=0, x=180
    vòng 2: chữ số 0 -> rev=0, x=18
    vòng 3: chữ số 8 -> rev=8, x=1
    vòng 4: chữ số 1 -> rev=81, x=0
  đảo 81:
    vòng 1: chữ số 1 -> rev=1, x=8
    vòng 2: chữ số 8 -> rev=18, x=0
  => reversed1=81, reversed2=18, so với num=1800: False


False

### 7.2. Trường hợp thành công và trường hợp `0`

In [16]:
trace_double(526)
print()
trace_double(0)

  đảo 526:
    vòng 1: chữ số 6 -> rev=6, x=52
    vòng 2: chữ số 2 -> rev=62, x=5
    vòng 3: chữ số 5 -> rev=625, x=0
  đảo 625:
    vòng 1: chữ số 5 -> rev=5, x=62
    vòng 2: chữ số 2 -> rev=52, x=6
    vòng 3: chữ số 6 -> rev=526, x=0
  => reversed1=625, reversed2=526, so với num=526: True

  đảo 0:
  đảo 0:
  => reversed1=0, reversed2=0, so với num=0: True


True

### 7.3. Số 0 ở **giữa** không bị mất

In [17]:
trace_double(1002)

  đảo 1002:
    vòng 1: chữ số 2 -> rev=2, x=100
    vòng 2: chữ số 0 -> rev=20, x=10
    vòng 3: chữ số 0 -> rev=200, x=1
    vòng 4: chữ số 1 -> rev=2001, x=0
  đảo 2001:
    vòng 1: chữ số 1 -> rev=1, x=200
    vòng 2: chữ số 0 -> rev=10, x=20
    vòng 3: chữ số 0 -> rev=100, x=2
    vòng 4: chữ số 2 -> rev=1002, x=0
  => reversed1=2001, reversed2=1002, so với num=1002: True


True

### 7.4. Bảng tổng hợp nhiều số — nhìn quy luật bằng mắt

Cột cuối cho thấy: **chỉ** những số khác 0 và tận cùng bằng 0 mới thất bại; số 0 ở **giữa** (như `1002`, `105`) không bị ảnh hưởng vì sau lần đảo đầu nó vẫn nằm giữa, không lên đầu.

In [18]:
print(f"{'num':>8} | {'reversed1':>9} | {'reversed2':>9} | bằng num? | tận cùng 0?")
for num in [0, 7, 10, 105, 120, 526, 1002, 1800, 90000, 123456]:
    r1 = int(str(num)[::-1]); r2 = int(str(r1)[::-1])
    print(f"{num:>8} | {r1:>9} | {r2:>9} | {str(r2 == num):>9} | {num % 10 == 0}")

     num | reversed1 | reversed2 | bằng num? | tận cùng 0?
       0 |         0 |         0 |      True | True
       7 |         7 |         7 |      True | False
      10 |         1 |         1 |     False | True
     105 |       501 |       105 |      True | False
     120 |        21 |        12 |     False | True
     526 |       625 |       526 |      True | False
    1002 |      2001 |      1002 |      True | False
    1800 |        81 |        18 |     False | True
   90000 |         9 |         9 |     False | True
  123456 |    654321 |    123456 |      True | False


## 8. Code nộp LeetCode (Python 3) — nhiều cách

### Cách A — Mô phỏng bằng chuỗi

In [19]:
class Solution:
    def isSameAfterReversals(self, num: int) -> bool:
        r1 = int(str(num)[::-1])
        r2 = int(str(r1)[::-1])
        return r2 == num

print([Solution().isSameAfterReversals(n) for n, _ in examples])

[True, False, True]


### Cách B — Mô phỏng bằng toán

In [20]:
class Solution:
    def isSameAfterReversals(self, num: int) -> bool:
        def rev(x: int) -> int:
            r = 0
            while x:
                r = r * 10 + x % 10
                x //= 10
            return r
        return rev(rev(num)) == num

print([Solution().isSameAfterReversals(n) for n, _ in examples])

[True, False, True]


`while x:` tương đương `while x != 0:` vì số 0 được coi là **False** trong ngữ cảnh điều kiện.

### Cách C — Quan sát chữ số cuối (lời giải chính, `O(1)`)

In [21]:
class Solution:
    def isSameAfterReversals(self, num: int) -> bool:
        return num == 0 or num % 10 != 0

print([Solution().isSameAfterReversals(n) for n, _ in examples])

[True, False, True]


### Cách D — Cùng quan sát, dạng chuỗi

`str.endswith("0")` kiểm tra ký tự cuối. Viết theo De Morgan: "không (khác 0 và tận cùng 0)".

In [22]:
class Solution:
    def isSameAfterReversals(self, num: int) -> bool:
        s = str(num)
        return s == "0" or not s.endswith("0")

print([Solution().isSameAfterReversals(n) for n, _ in examples])

[True, False, True]


### Cách E — Một phép so sánh: "số 0 cuối bị mất" ⇔ `reversed1` có ít chữ số hơn

Đảo một lần rồi so **số chữ số**: nếu `len(str(reversed1)) == len(str(num))` thì không mất gì.

In [23]:
class Solution:
    def isSameAfterReversals(self, num: int) -> bool:
        return len(str(int(str(num)[::-1]))) == len(str(num))

print([Solution().isSameAfterReversals(n) for n, _ in examples])

[True, False, True]


## 9. Vì sao thuật toán đúng?

Gọi chuỗi chữ số của `num` là `s` (không có số 0 đầu, trừ khi `num = 0`).

**Trường hợp `num = 0`:** đảo 0 → 0 → 0. Đúng `True`.

**Trường hợp `num > 0` và chữ số cuối `≠ 0`:** `s[::-1]` bắt đầu bằng chữ số khác 0 ⇒ `int` không xóa gì ⇒ `reversed1` có chuỗi đúng bằng `s[::-1]`. Đảo lần nữa: `(s[::-1])[::-1] = s`, và `s` bắt đầu khác 0 ⇒ `reversed2 = num`. ⇒ `True`.

**Trường hợp `num > 0` và chữ số cuối `= 0`:** `s[::-1]` có ít nhất một số 0 đầu ⇒ `reversed1` có **ít chữ số hơn** `num` (ít nhất 1). Đảo không làm tăng số chữ số ⇒ `reversed2` có ít chữ số hơn `num` ⇒ `reversed2 ≠ num`. ⇒ `False`.

Ba trường hợp phủ toàn bộ miền ⇒ công thức `num == 0 or num % 10 != 0` đúng.

## 10. Phân tích Time / Space Complexity

Gọi `d` = số chữ số của `num` (`d ≤ 7` vì `num ≤ 10^6`), `d = O(log₁₀ num)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| A. Mô phỏng chuỗi | `O(d)` | `O(d)` | đúng nghĩa đen của đề |
| B. Mô phỏng toán | `O(d)` | `O(1)` | luyện `% 10`, `// 10` |
| C. Chữ số cuối | `O(1)` | `O(1)` | lời giải chính |
| D. `endswith` | `O(d)` (tạo chuỗi) | `O(d)` | dễ đọc |
| E. So số chữ số | `O(d)` | `O(d)` | cách nhìn khác cùng bản chất |

### Đo thời gian thực tế trên toàn miền

Cả ba đều nhanh với một lần gọi, nhưng khi gọi `10^6` lần thì hằng số khác biệt rõ: công thức `O(1)` không tạo chuỗi, không lặp.

In [24]:
import time
for name, f in [("mô phỏng chuỗi", is_same_simulate_str), ("mô phỏng toán", is_same_simulate_math), ("chữ số cuối", is_same_after_reversals)]:
    t = time.perf_counter()
    for n in range(0, 200_000):
        f(n)
    print(f"{name:15}: {(time.perf_counter() - t) * 1000:.1f} ms cho 200.000 lần gọi")

mô phỏng chuỗi : 81.8 ms cho 200.000 lần gọi


mô phỏng toán  : 172.0 ms cho 200.000 lần gọi
chữ số cuối    : 18.2 ms cho 200.000 lần gọi


## 11. Kiểm thử: mẫu đề, biên, **vét cạn toàn miền**

Miền input chỉ có `10^6 + 1` giá trị nên ta có thể so lời giải `O(1)` với mô phỏng trên **toàn bộ** miền — đây là kiểm thử mạnh nhất!

In [25]:
def sol_A(n): return is_same_simulate_str(n)
def sol_B(n): return is_same_simulate_math(n)
def sol_C(n): return is_same_after_reversals(n)
def sol_D(n): s = str(n); return s == "0" or not s.endswith("0")
def sol_E(n): return len(str(int(str(n)[::-1]))) == len(str(n))

SOLS = [sol_A, sol_B, sol_C, sol_D, sol_E]

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:24} PASS ({actual})")

tests = examples + [(1, True), (10, False), (100, False), (101, True), (1000000, False), (999999, True), (1002, True), (20, False)]
for n, exp in tests:
    for f in SOLS:
        check(f"num={n} {f.__name__}", f(n), exp)

num=526 sol_A            PASS (True)
num=526 sol_B            PASS (True)
num=526 sol_C            PASS (True)
num=526 sol_D            PASS (True)
num=526 sol_E            PASS (True)
num=1800 sol_A           PASS (False)
num=1800 sol_B           PASS (False)
num=1800 sol_C           PASS (False)
num=1800 sol_D           PASS (False)
num=1800 sol_E           PASS (False)
num=0 sol_A              PASS (True)
num=0 sol_B              PASS (True)
num=0 sol_C              PASS (True)
num=0 sol_D              PASS (True)
num=0 sol_E              PASS (True)
num=1 sol_A              PASS (True)
num=1 sol_B              PASS (True)
num=1 sol_C              PASS (True)
num=1 sol_D              PASS (True)
num=1 sol_E              PASS (True)
num=10 sol_A             PASS (False)
num=10 sol_B             PASS (False)
num=10 sol_C             PASS (False)
num=10 sol_D             PASS (False)
num=10 sol_E             PASS (False)
num=100 sol_A            PASS (False)
num=100 sol_B            PA

In [26]:
# Vét cạn: so cách C với mô phỏng chuỗi trên [0, 10^6]
mismatch = [n for n in range(0, 10**6 + 1) if sol_C(n) != sol_A(n)]
print("Số giá trị lệch trên toàn miền:", len(mismatch))
assert not mismatch
print("Số lượng num cho True:", sum(sol_C(n) for n in range(0, 10**6 + 1)))

Số giá trị lệch trên toàn miền: 0
Số lượng num cho True: 900001


## 12. Lỗi người mới hay gặp — code sai, vì sao sai, cách sửa

### Lỗi 1: Quên trường hợp `num = 0`

In [27]:
def wrong_zero(num):
    return num % 10 != 0          # SAI với num = 0

print("Sai :", wrong_zero(0), "| Đúng:", sol_C(0))

Sai : False | Đúng: True


### Lỗi 2: Đảo bằng chuỗi nhưng quên `int(...)` nên giữ lại số 0 đầu

In [28]:
def wrong_keep_zeros(num):
    r1 = str(num)[::-1]           # '0081' - vẫn còn số 0
    r2 = r1[::-1]                 # '1800'
    return int(r2) == num         # SAI: không mô phỏng việc bỏ số 0

print("Sai :", wrong_keep_zeros(1800), "| Đúng:", sol_C(1800))

Sai : True | Đúng: False


**Vì sao sai:** đề nói số 0 đầu **không được giữ**. Phải `int(...)` sau **mỗi** lần đảo.

### Lỗi 3: Dùng `/` thay `//` khi đảo bằng toán

In [29]:
def wrong_div(x):
    rev, steps = 0, 0
    while x > 0 and steps < 5:      # giới hạn để không chạy quá lâu
        rev = rev * 10 + x % 10
        x = x / 10                  # SAI: ra số thực
        steps += 1
    return rev

print("Sai :", wrong_div(526), "| Đúng:", reverse_math(526))

Sai : 63131.312600000005 | Đúng: 625


### Lỗi 4: So sánh `reversed1 == num` (chỉ đảo một lần)

In [30]:
def wrong_single(num):
    return int(str(num)[::-1]) == num   # SAI: đây là kiểm tra palindrome

print("Sai :", wrong_single(526), "| Đúng:", sol_C(526))

Sai : False | Đúng: True


### Lỗi 5: Viết điều kiện ngược bằng `and`

`num == 0 and num % 10 != 0` không bao giờ đúng (nếu `num == 0` thì `num % 10 == 0`). Khi gộp các trường hợp "đặc biệt **hoặc** bình thường", dùng `or`. Mẹo kiểm tra: thử ngay với 3 ví dụ của đề trước khi nộp.

In [31]:
def wrong_and(num):
    return num == 0 and num % 10 != 0      # SAI: luôn False

print([wrong_and(n) for n, _ in examples], "| Đúng:", [sol_C(n) for n, _ in examples])

[False, False, False] | Đúng: [True, False, True]


## 13. Mẹo nhận dạng pattern

- Đề bắt "mô phỏng một quá trình" trên số nhỏ → thử **in ra nhiều trường hợp** để tìm **quy luật** (ở đây: các số thất bại đều tận cùng bằng 0).
- "Đảo chữ số" → nhớ: **số 0 cuối** biến thành **số 0 đầu** và bị mất.
- Miền input nhỏ (≤ 10^6, ≤ 10^7) → **vét cạn** để kiểm chứng công thức.
- Bộ đôi `x % 10` / `x // 10` là nền tảng cho mọi bài xử lý chữ số.

Bài liên quan: 7 Reverse Integer, 9 Palindrome Number, 2442 Count Number of Distinct Integers After Reverse Operations.

## 14. Biến thể

### 14.1. Đảo `k` lần thì sao?

Sau lần đảo đầu, các số 0 cuối đã mất; từ đó mọi lần đảo đều giữ nguyên số chữ số. Kết quả: với `k ≥ 2` chẵn, câu trả lời giống bài gốc; với `k` lẻ, ta cần `num` là **palindrome**.

In [32]:
def after_k_reversals(num, k):
    for _ in range(k):
        num = int(str(num)[::-1])
    return num

for k in [1, 2, 3, 4]:
    print(f"k={k}: 1800 -> {after_k_reversals(1800, k)}, 526 -> {after_k_reversals(526, k)}, 121 -> {after_k_reversals(121, k)}")

k=1: 1800 -> 81, 526 -> 625, 121 -> 121
k=2: 1800 -> 18, 526 -> 526, 121 -> 121
k=3: 1800 -> 81, 526 -> 625, 121 -> 121
k=4: 1800 -> 18, 526 -> 526, 121 -> 121


### 14.2. Số nguyên **âm** (giữ dấu)

In [33]:
def is_same_signed(num):
    return num == 0 or abs(num) % 10 != 0

print(is_same_signed(-526), is_same_signed(-1800))

True False


### 14.3. Đếm số lượng `num` trong `[lo, hi]` thỏa mãn — dùng công thức đếm bội của 10

In [34]:
def count_same(lo, hi):
    # số bội của 10 trong [lo, hi], trừ đi (nhưng 0 vẫn đúng)
    multiples = hi // 10 - (lo - 1) // 10 if lo > 0 else hi // 10 + 1
    zero_inside = 1 if lo <= 0 <= hi else 0
    return (hi - lo + 1) - multiples + zero_inside

print(count_same(0, 10**6), sum(sol_C(n) for n in range(0, 10**6 + 1)))
print(count_same(5, 47), sum(sol_C(n) for n in range(5, 48)))

900001 900001
39 39


### 14.4. Cơ số khác: đảo chữ số trong hệ nhị phân

Ý tưởng tổng quát hóa: trong cơ số `b`, đảo hai lần giữ nguyên số ⇔ `num == 0` hoặc chữ số cuối (`num % b`) khác 0. Với nhị phân: số **lẻ** (bit cuối = 1) hoặc 0.

In [35]:
def reverse_base(x, base):
    r = 0
    while x:
        r = r * base + x % base
        x //= base
    return r

for num in [0, 5, 6, 12, 13]:
    same = reverse_base(reverse_base(num, 2), 2) == num
    print(f"{num:>3} = {num:>5b}b -> đảo 2 lần giữ nguyên? {same} | công thức: {num == 0 or num % 2 != 0}")

  0 =     0b -> đảo 2 lần giữ nguyên? True | công thức: True
  5 =   101b -> đảo 2 lần giữ nguyên? True | công thức: True
  6 =   110b -> đảo 2 lần giữ nguyên? False | công thức: False
 12 =  1100b -> đảo 2 lần giữ nguyên? False | công thức: False
 13 =  1101b -> đảo 2 lần giữ nguyên? True | công thức: True


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

Đáp án ở **mục 17**.

### Bài 1 — Đếm số chữ số của `num` bằng toán (không dùng `str`), `0` có 1 chữ số

In [36]:
def digit_count(num):
    # TODO: viết code ở đây
    pass

# assert digit_count(0) == 1 and digit_count(1800) == 4

### Bài 2 — Đếm số chữ số 0 ở **cuối** `num` (`num > 0`)

In [37]:
def trailing_zeros(num):
    # TODO: viết code ở đây
    pass

# assert trailing_zeros(12300) == 2 and trailing_zeros(7) == 0

### Bài 3 — Tự viết lại lời giải #2119 (không nhìn mục 6)

In [38]:
def is_same_ex(num):
    # TODO: viết code ở đây
    pass

# assert is_same_ex(526) and not is_same_ex(1800) and is_same_ex(0)

### Bài 4 — Biến thể: trả về **hiệu** `num - reversed2`

In [39]:
def loss_after_double(num):
    # TODO: viết code ở đây
    pass

# assert loss_after_double(1800) == 1782 and loss_after_double(526) == 0

### Bài 5 — Tự tạo test làm lộ **Lỗi 2** (giữ số 0 đầu)

In [40]:
my_tests = [
    # TODO: các num mà wrong_keep_zeros(num) != sol_C(num)
]
# for n in my_tests:
#     assert wrong_keep_zeros(n) != sol_C(n)

### Bài 6 — Mini challenge: Reverse Integer có giới hạn 32-bit (LeetCode 7)
Đảo `x` (có thể âm); nếu kết quả ngoài `[-2^31, 2^31 - 1]` trả `0`.

In [41]:
def reverse_32(x):
    # TODO: viết code ở đây
    pass

# assert reverse_32(123) == 321 and reverse_32(-120) == -21 and reverse_32(1534236469) == 0

## 16. Gợi ý ba mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | chia 10 tới khi hết | đếm số vòng | xử lý riêng `num == 0` |
| 2 | xét chữ số cuối | `while num % 10 == 0` | mỗi vòng `num //= 10` |
| 3 | số 0 cuối bị mất | trừ `num = 0` | `num == 0 or num % 10 != 0` |
| 4 | mô phỏng hai lần | `int(str(...)[::-1])` | trả `num - r2` |
| 5 | lỗi không bỏ số 0 | chỉ lộ khi num tận cùng 0 | `10`, `1800` |
| 6 | lấy dấu riêng | đảo `abs(x)` | kiểm tra giới hạn sau khi gắn dấu |

## 17. Đáp án tham khảo

In [42]:
def digit_count(num):
    if num == 0:
        return 1
    c = 0
    while num > 0:
        num //= 10
        c += 1
    return c

def trailing_zeros(num):
    c = 0
    while num % 10 == 0:
        num //= 10
        c += 1
    return c

def is_same_ex(num):
    return num == 0 or num % 10 != 0

def loss_after_double(num):
    r1 = int(str(num)[::-1])
    r2 = int(str(r1)[::-1])
    return num - r2

my_tests = [10, 1800, 1000000]

def reverse_32(x):
    sign = -1 if x < 0 else 1
    r = sign * int(str(abs(x))[::-1])
    return r if -2**31 <= r <= 2**31 - 1 else 0

assert digit_count(0) == 1 and digit_count(1800) == 4
assert trailing_zeros(12300) == 2 and trailing_zeros(7) == 0
assert is_same_ex(526) and not is_same_ex(1800) and is_same_ex(0)
assert loss_after_double(1800) == 1782 and loss_after_double(526) == 0
for n in my_tests:
    assert wrong_keep_zeros(n) != sol_C(n)
assert reverse_32(123) == 321 and reverse_32(-120) == -21 and reverse_32(1534236469) == 0
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


**Giải thích bằng lời:**
- Bài 1: mỗi lần `// 10` bỏ một chữ số; số 0 phải xử lý riêng vì vòng lặp không chạy.
- Bài 2: `trailing_zeros(num) > 0` chính là điều kiện "thất bại" của bài gốc (với `num > 0`).
- Bài 3: công thức `O(1)`.
- Bài 4: `1800 → 81 → 18`, mất `1782`.
- Bài 5: `wrong_keep_zeros` luôn trả `True` vì đảo chuỗi hai lần là chính nó → sai đúng ở những số tận cùng bằng 0.
- Bài 6: tách dấu, đảo phần trị tuyệt đối, kiểm tra tràn 32-bit.

## 18. Tổng kết — template + checklist

**Template:**
```python
return num == 0 or num % 10 != 0
```

**Tư duy:** mô phỏng vài trường hợp → tìm quy luật → chứng minh → kiểm chứng vét cạn (khi miền nhỏ).

**Checklist trước khi submit:**
- [ ] `num = 0` trả `True`.
- [ ] Số tận cùng bằng 0 (khác 0) trả `False`.
- [ ] Nếu mô phỏng: `int(...)` sau **mỗi** lần đảo; dùng `//` không dùng `/`.
- [ ] Đảo **hai** lần, không phải một.
- [ ] Đúng tên method `isSameAfterReversals`.